# Re-scoring on New Data — `amygda_ops_risk_score`

## What this notebook does

Use this notebook when you want to **score a new batch of logs using a model you already trained**.

You have already run `01_labelling.ipynb` and `02_risk_score.ipynb` at least once.
Those notebooks produced a `complete_model_ses-xxx.zip` — that zip contains:
- The trained labelling model (knows your systems and subsystems)
- The calibrated risk thresholds (knows what counts as normal vs elevated for each subsystem)

This notebook takes that zip and a **new log file** and produces a fresh `risk_scores.parquet`.
**No re-training. No re-labelling. No LLM calls.**

## What you need

| Item | Where to find it |
|------|------------------|
| `complete_model_ses-xxx.zip` | Downloaded by `generate_risk_scores()` in `02_risk_score.ipynb` |
| New log file (.csv or .xlsx) | Your latest operational data |
| Your `api_key` | Same permanent key from the labelling notebook |

## How it works

`import_model()` detects the zip type automatically:
- `complete_model_ses-xxx.zip` → skips training (Steps 9–10), goes straight to `configure_generation`
- `trained_labelling_model_ses-xxx.zip` → requires training (Steps 9–12)

For re-scoring, always use the **complete model zip**.

In [ ]:
# ── Cell 1: Install and import ────────────────────────────────────────────────
# Run once at the start of every session.
# --upgrade ensures you always have the latest version of the SDK.
#!pip install --upgrade git+https://github.com/amygda/amygda_ops_risk_score_sdk.git

import json, os
from amygda_ops_risk_score import OpsRiskClient, SessionConfig
from amygda_ops_risk_score import helpers

print("SDK ready.")

In [ ]:
# ── Connect and wait for the API ────────────────────────────────────
client = OpsRiskClient()
client.wait_until_ready()
print("API is ready.")

In [ ]:
# ── Cell 2: Session setup — run once per re-score run ────────────────────────
# ⚠  KERNEL RESTARTED? Use Cell 4 instead — do not re-run this cell.
#
# api_key is the same permanent key from 01_labelling.ipynb.
# INTERNAL TESTING — two keys are already live: yk-amygda-dev-normal ($20) or yk-amygda-dev-low ($4).
# PRODUCTION: get your key from the Amygda portal (https://portal.amygda.io — coming soon).
# ─────────────────────────────────────────────────────────────────────────────

# ── Your complete model zip ───────────────────────────────────────────────────
# Option A: load automatically from the last generation artifact
PREV_ARTIFACT_DIR = "my_artifacts/risk_score/"   # ← adjust if you used a different path
with open(f"{PREV_ARTIFACT_DIR}generate_risk_scores.json") as f:
    COMPLETE_ZIP = json.load(f)["zip_path"]

# Option B: paste the path directly
# ZIP_PATH = "my_outputs/complete_model_ses-xxxx.zip"  ← from previous 02_risk_score run

print(f"Using model zip: {COMPLETE_ZIP}")

# ── Your API key ──────────────────────────────────────────────────────────────
with open("my_artifacts/labelling/api_key.txt") as f:
    API_KEY = f.read().strip()
# Or paste directly: API_KEY = "yk-..."

# ── Artifact directory for this run ───────────────────────────────────────────
# Change this for each re-score run to keep results organised.
ARTIFACT_DIR = "my_artifacts/rescore/"

# ── Open a new session ────────────────────────────────────────────────────────
config  = SessionConfig(name="rescore-run")
session = client.open_session(api_key=API_KEY, config=config, artifact_dir=ARTIFACT_DIR)

AUTH_ID = session.auth_id
print(f"⚠  SAVE THIS AUTH_ID (needed if kernel restarts): {AUTH_ID}")

# ── Import the complete model zip ─────────────────────────────────────────────
# This loads the trained labelling model AND the calibrated thresholds.
# The server detects the thresholds and skips training automatically.
import_result = session.import_model(COMPLETE_ZIP)
print()
print(f"next_step: {import_result['next_step']}")
if import_result["next_step"] == "configure_generation":
    print("  ✓ Thresholds detected — training skipped. Ready to score new data.")
else:
    print("  ✗ This zip does not contain thresholds — use a complete_model_ses-xxx.zip,")
    print("    not a trained_labelling_model_ses-xxx.zip.")

In [ ]:
# ── Cell 3: Upload new logs ───────────────────────────────────────────────────
# The new log file must have the same column names as the file you originally
# trained on — same log text column, same timestamp column, same asset ID column.
# The data can cover any date range; it does not need to overlap training data.
# ─────────────────────────────────────────────────────────────────────────────

NEW_LOG_FILE = "../sample_data/risk_score_prediction.csv"   # ← edit: path to your new log file
DATE_FORMAT  = "infer"               # "infer" auto-detects; or provide strftime format
SHEET_NAME   = None                  # XLSX only — None = auto-detect

generation_config_result = session.configure_generation(
    file_path=NEW_LOG_FILE,
    date_format=DATE_FORMAT,
    sheet_name=SHEET_NAME,
)
print(json.dumps(generation_config_result, indent=2))

In [ ]:
# ── Cell 4: Kernel restarted — run Cell 1 then this cell ─────────────────────
# Reconnects to your existing session without opening a new one.
# The model is already loaded on the server — no need to re-import the zip.
# If get_session() raises SessionNotFoundError, the session expired → use Cell 2.
# ─────────────────────────────────────────────────────────────────────────────

AUTH_ID      = "ses-paste-here"     # ← paste your saved AUTH_ID
ARTIFACT_DIR = "my_artifacts/rescore/" # ← same path as Cell 2

with open("my_artifacts/labelling/api_key.txt") as f:
    API_KEY = f.read().strip()

session = client.get_session(AUTH_ID, artifact_dir=ARTIFACT_DIR)
status  = session.status()
print(json.dumps(status, indent=2))
print()
print("Step states — DONE: skip it | RUNNING: wait | FAILED: re-run | NONE: proceed")

In [ ]:
# ── Cell 5: Generate risk scores ⚠️ ONE-WAY DOOR ─────────────────────────────
# Classifies new logs using the trained model, applies the calibrated thresholds,
# downloads the complete_model zip, extracts risk_scores.parquet, wipes the session.
#
# Save the new complete_model zip — use it as COMPLETE_ZIP for the next re-score run.
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR = "my_outputs/rescore/"   # ← directory to save the complete model zip into

generation_result = session.generate_risk_scores(OUTPUT_DIR, timeout=3600)

print(f"Complete model zip:  {generation_result['zip_path']}")
print(f"Risk scores parquet: {generation_result.get('parquet_path')}")
print(f"Thresholds:          {generation_result.get('thresholds_path')}")
print(f"Hierarchy:           {generation_result.get('hierarchy_path')}")
print("Session wiped automatically.")
print()
print("Next re-score run: use the complete_model zip above as your COMPLETE_ZIP in Cell 2.")

## Visualise risk scores

All plot helpers accept a file path or a DataFrame directly.  The file path form works
even after a kernel restart — no need to re-run `run_generation()`.

```python
# After kernel restart — no re-run needed:
helpers.plot_asset_risk_ranking(f"{ARTIFACT_DIR}risk_scores.parquet")

# If you have the DataFrame already loaded:
import pandas as pd
risk_df = pd.read_parquet(f"{ARTIFACT_DIR}risk_scores.parquet")
helpers.plot_asset_risk_ranking(risk_df)
```

In [ ]:
# ── Plot from the parquet file (also works after kernel restart) ──────────────
parquet_path = generation_result.get("parquet_path") or f"{ARTIFACT_DIR}risk_scores.parquet"

# Bar chart: top-20 assets by mean operational_risk
helpers.plot_asset_risk_ranking(
    parquet_path,
    aggregation="max",
    top_n=20,
    title="Asset Operational Risk Scores",
)

# Histogram: distribution of risk scores across all records
helpers.plot_risk_score_distribution(
    parquet_path,
    title="Risk Score Distribution",
)

In [ ]:
# ── Load parquet for custom analysis ─────────────────────────────────────────
import pandas as pd

parquet_path = generation_result.get("parquet_path") or f"{ARTIFACT_DIR}risk_scores.parquet"
risk_df = pd.read_parquet(parquet_path)

print(f"Risk scores: {len(risk_df):,} rows, {risk_df['asset_id'].nunique()} assets")
print(f"Columns: {list(risk_df.columns[:10])} ... ({len(risk_df.columns)} total)")
risk_df.head()

## Visual Analysis

All five helpers below accept a file path **or** a DataFrame — file paths work after a kernel restart without re-running `run_generation()`.

```python
parquet_path         = generation_result.get("parquet_path")          or f"{ARTIFACT_DIR}risk_scores.parquet"
classified_logs_path = generation_result.get("classified_logs_path")  or f"{ARTIFACT_DIR}classified_logs.parquet"
```

### Helpers overview

| Helper | Description | Mode |
|--------|-------------|------|
| `plot_risk_heatmap` | Heatmap: all assets × all dates | Both |
| `plot_daily_risk_snapshot` | Tile panel for one specific date | Both |
| `plot_asset_risk_over_time` | Heatmap: all risk types × all dates for one asset | Both |
| `plot_subsystem_log_activity` | Stacked bar of log code counts per subsystem | Fixed-log only |
| `get_logs_for_subsystem` | Raw log text table for a subsystem + date window | Free-text only |

In [ ]:
# ── Multi-Asset Risk Heatmap Over Time ───────────────────────────────────────
# Rows = assets, columns = dates, colour = chosen risk metric.
#
# metric= accepts:
#   "operational_risk"           — overall aggregate score (default)
#   "{system}_system_risk"       — e.g. "conveyor_handling_system_risk"
#                                   run helpers.list_risk_metrics(parquet_path) to see all
#
parquet_path = generation_result.get("parquet_path") or f"{ARTIFACT_DIR}risk_scores.parquet"

# See every valid metric value for this parquet:
print("Available metrics:")
for m in helpers.list_risk_metrics(parquet_path):
    print(f"  {m}")
print()

helpers.plot_risk_heatmap(
    parquet_path,
    metric="operational_risk",          # ← swap to any {system}_system_risk column
    # asset_ids=["K204", "K205"],       # optional: limit to specific assets
    title="Multi-Asset Operational Risk Heatmap",
)

In [ ]:
# ── Risk Scores by Date ───────────────────────────────────────────────────────
# Tile panel for a single date: one large op-risk tile per asset,
# with system-level tiles below each asset.
# Colour: blue (0-33), yellow (33-66), red (66-100).

SCORE_DATE = "2025-01-09"          # ← edit: the date you want to inspect
MIN_SCORE  = 0.0                   # ← hide assets below this operational_risk threshold

helpers.plot_daily_risk_snapshot(
    parquet_path,
    date=SCORE_DATE,
    min_score=MIN_SCORE,
    # asset_ids=["asset-001"],     # optional filter
    title="Risk Scores by Date",
)

In [ ]:
# ── Single Asset Risk Analysis ────────────────────────────────────────────────
# Heatmap for one asset: rows = risk types (Operational Risk at top,
# then each system), columns = dates.

ASSET_TO_INSPECT = "K236"    # ← edit: the asset you want to drill into

helpers.plot_asset_risk_over_time(
    parquet_path,
    asset_id=ASSET_TO_INSPECT,
    title=f"Risk Analysis — {ASSET_TO_INSPECT}",
)

helpers.plot_asset_risk_over_time(
    parquet_path,
    asset_id=ASSET_TO_INSPECT,
    weighted=True,
    model_config=f"{ARTIFACT_DIR}model_config.json",
    title=f"Risk Analysis — Weighted {ASSET_TO_INSPECT}",
)

In [ ]:
# ── System-level heatmap for one asset ───────────────────────────────────────────
# Rows = subsystem risk scores, columns = dates.
# Edit SYSTEM to any system name from the hierarchy (snake_case).

SYSTEM = "baggage_handling"   # ← edit: system name from hierarchy

helpers.plot_asset_system_over_time(
    parquet_path,
    asset_id=ASSET_TO_INSPECT,
    system=SYSTEM,
    title=f"{SYSTEM.replace('_',' ').title()} — {ASSET_TO_INSPECT}",
)

helpers.plot_asset_system_over_time(
    parquet_path,
    asset_id=ASSET_TO_INSPECT,
    system=SYSTEM,
    weighted=True,
    model_config=f"{ARTIFACT_DIR}model_config.json",
    title=f"{SYSTEM.replace('_',' ').title()} — Weighted {ASSET_TO_INSPECT}",
)

In [ ]:
# ── Load model_config.json for inspection ─────────────────────────────────────────
import json as _json

model_config_path = generation_result.get("model_config_path") or f"{ARTIFACT_DIR}model_config.json"

with open(model_config_path) as _f:
    model_config = _json.load(_f)

print("model_config.json loaded.")

In [ ]:
# ── Operational risk breakdown — contribution of each system for one asset ───
helpers.plot_system_contributions(
    parquet_path,
    asset_id=ASSET_TO_INSPECT,
    model_config=model_config_path,
)

In [ ]:
# ── Risk Breakdown — Asset + Date ────────────────────────────────────────────
# Full operational risk + every system and subsystem score for one asset on one date.
# No need to know any system or subsystem names in advance.
# If the exact date is not in the parquet, the closest available date is used.

BREAKDOWN_ASSET = "K397"        # ← edit: asset to inspect
BREAKDOWN_DATE  = "2025-09-19"  # ← edit: date to inspect ("YYYY-MM-DD")

breakdown = helpers.get_risk_breakdown(parquet_path, BREAKDOWN_ASSET, BREAKDOWN_DATE)

import json
print(json.dumps(breakdown, indent=2))

# ── Flat DataFrame view ───────────────────────────────────────────────────────
import pandas as pd

rows = [{"level": "operational", "name": "operational_risk",
         "score": breakdown["operational_risk"]}]
for sys_name, score in breakdown["systems"].items():
    rows.append({"level": "system", "name": sys_name, "score": score})
for pair, score in breakdown["subsystems"].items():
    rows.append({"level": "subsystem", "name": pair, "score": score})

df_breakdown = (
    pd.DataFrame(rows)
    .sort_values(["level", "score"], ascending=[True, False])
    .reset_index(drop=True)
)
#display(df_breakdown)

In [ ]:
# ── Log Occurrences — Subsystem Breakdown ────────────────────────────────────
# Auto-detects mode from the source file:
#   Free-text mode  → pass classified_logs.parquet
#                     shows daily count of log entries for the subsystem
#   Fixed-log mode  → pass risk_scores.parquet + logs_mapping
#                     shows stacked bar of individual log-code daily counts
import os
parquet_path         = generation_result.get("parquet_path")         or f"{ARTIFACT_DIR}risk_scores.parquet"
classified_logs_path = generation_result.get("classified_logs_path") or f"{ARTIFACT_DIR}classified_logs.parquet"
logs_mapping_path    = generation_result.get("logs_mapping_path")    or f"{ARTIFACT_DIR}logs_by_system_subsystem.json"

ASSET_ID   = "K397"                      # ← edit
SYSTEM     = "motion_control"            # ← edit: system name from the hierarchy
SUBSYSTEM  = "datum_homing"             # ← edit: subsystem name
END_DATE   = "2025-09-19"               # ← edit: end of the window
DAYS_BACK  = 60                          # rolling window in days
LOG_COLUMN = "cleaned_event_details"     # ← free-text mode only; ignored in fixed-log mode

if os.path.exists(classified_logs_path):
    # Free-text mode: classified_logs.parquet exists
    # helpers.plot_subsystem_log_activity(
    #     classified_logs_path,
    #     asset_id=ASSET_ID, system=SYSTEM, subsystem=SUBSYSTEM,
    #     date=END_DATE, days_back=DAYS_BACK, log_column=LOG_COLUMN,
    #     title=f"Log Occurrences — {ASSET_ID} / {SYSTEM} / {SUBSYSTEM}",
    # )
    helpers.plot_subsystem_log_activity(
    classified_logs_path,
    asset_id=ASSET_ID, system=SYSTEM, subsystem=SUBSYSTEM,
    date=END_DATE, days_back=DAYS_BACK, log_column=LOG_COLUMN,
    risk_source=parquet_path,   # <-- add this
    title=f"Log Occurrences — {ASSET_ID} / {SYSTEM} / {SUBSYSTEM}",
)
elif os.path.exists(parquet_path) and os.path.exists(logs_mapping_path):
    # Fixed-log mode: use risk_scores.parquet + logs_by_system_subsystem.json
    # helpers.plot_subsystem_log_activity(
    #     parquet_path,
    #     asset_id=ASSET_ID, system=SYSTEM, subsystem=SUBSYSTEM,
    #     date=END_DATE, days_back=DAYS_BACK,
    #     logs_mapping=logs_mapping_path,
    #     title=f"Log Occurrences — {ASSET_ID} / {SYSTEM} / {SUBSYSTEM}",
    # )
    helpers.plot_subsystem_log_activity(
    parquet_path,
    asset_id=ASSET_ID, system=SYSTEM, subsystem=SUBSYSTEM,
    date=END_DATE, days_back=DAYS_BACK, log_column=LOG_COLUMN,
    risk_source=parquet_path,   # <-- add this
    logs_mapping=logs_mapping_path,
    title=f"Log Occurrences — {ASSET_ID} / {SYSTEM} / {SUBSYSTEM}",)
else:
    print("Source files not found. Check ARTIFACT_DIR and re-run generation.")

In [ ]:
# ── Raw Log Table (free-text mode only) ──────────────────────────────────────
# Returns a DataFrame of classified raw log entries for a chosen asset /
# system / subsystem within a date window.
# classified_logs.parquet is extracted to artifact_dir by run_generation()
# only when is_free_text=True was used in the labelling notebook.

import os

classified_logs_path = generation_result.get("classified_logs_path") or f"{ARTIFACT_DIR}classified_logs.parquet"

ASSET_ID   = "K236"                      # ← edit
SYSTEM     = "motion_control"            # ← edit: system name from the hierarchy
SUBSYSTEM  = "axis_control"             # ← edit: subsystem name
END_DATE   = "2025-12-09"               # ← edit: end of the window
DAYS_BACK  = 90                          # rolling window in days
LOG_COLUMN = "cleaned_event_details"     # ← match the log_column used in configure()

if not os.path.exists(classified_logs_path):
    print("classified_logs.parquet not available — this helper is for free-text mode only.")
    print(f"Expected at: {classified_logs_path}")
else:
    logs_df = helpers.get_logs_for_subsystem(
        classified_logs_path,
        asset_id=ASSET_ID,
        system=SYSTEM,
        subsystem=SUBSYSTEM,
        date=END_DATE,
        days_back=DAYS_BACK,
        log_column=LOG_COLUMN,
    )
    print(f"{len(logs_df)} log entries found")
    display(logs_df)